In [2]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf

# Install linearmodels if not already installed
try:
    from linearmodels.panel import PanelOLS, RandomEffects
    from linearmodels.iv import IV2SLS
except ImportError:
    !pip install linearmodels
    from linearmodels.panel import PanelOLS, RandomEffects
    from linearmodels.iv import IV2SLS

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 20)

DATA_PATH = "/content/sa_spatial_mismatch_panel.csv"
df = pd.read_csv(DATA_PATH)

# -----------------------------------------------------------------
# 0. Prep
# -----------------------------------------------------------------
df["formal"] = (df["employment_status"] == "Formal").astype(int)
df["informal"] = (df["employment_status"] == "Informal").astype(int)
df["unemployed"] = (df["employment_status"] == "Unemployed").astype(int)

# distance terciles (proxy for degree of spatial mismatch exposure)
df["distance_tercile"] = pd.qcut(df["distance_cbd_km"], 3,
                                  labels=["Near CBD", "Mid distance", "Far (peripheral)"])

print("=" * 80)
print("1. DESCRIPTIVE STATISTICS")
print("=" * 80)
print(df[["distance_cbd_km", "transport_cost_month_rand",
          "transport_cost_share_income", "education_years",
          "monthly_earnings_rand"]].describe().T)

print("\nEmployment status shares by wave:")
print(pd.crosstab(df["wave"], df["employment_status"], normalize="index").round(3))

print("\nEmployment status shares by distance tercile:")
print(pd.crosstab(df["distance_tercile"], df["employment_status"], normalize="index").round(3))

print("\nEmployment status shares by race (spatial-mismatch legacy check):")
print(pd.crosstab(df["race"], df["employment_status"], normalize="index").round(3))

# -----------------------------------------------------------------
# 2. Markov transition matrices
# -----------------------------------------------------------------
print("\n" + "=" * 80)
print("2. MARKOV TRANSITION MATRICES (row = t-1 status, col = t status)")
print("=" * 80)

trans = df.dropna(subset=["prev_employment_status"])

def transition_matrix(frame, label):
    tm = pd.crosstab(frame["prev_employment_status"], frame["employment_status"],
                      normalize="index").round(3)
    tm = tm.reindex(index=["Unemployed", "Informal", "Formal"],
                     columns=["Unemployed", "Informal", "Formal"])
    print(f"\n-- {label} --")
    print(tm)
    return tm

transition_matrix(trans, "Overall")
for tercile in ["Near CBD", "Mid distance", "Far (peripheral)"]:
    transition_matrix(trans[trans["distance_tercile"] == tercile], f"Distance tercile: {tercile}")

# -----------------------------------------------------------------
# 3. Multinomial logit: static determinants of employment status
# -----------------------------------------------------------------
print("\n" + "=" * 80)
print("3. MULTINOMIAL LOGIT — employment_status ~ distance + transport cost + controls")
print("=" * 80)

mnl_df = df.copy()
mnl_df["status_cat"] = pd.Categorical(mnl_df["employment_status"],
                                       categories=["Unemployed", "Informal", "Formal"])
y = mnl_df["status_cat"].cat.codes  # 0=Unemployed,1=Informal,2=Formal
X = pd.get_dummies(mnl_df[["distance_cbd_km", "transport_cost_share_income",
                            "education_years", "age", "race", "gender", "province"]],
                    columns=["race", "gender", "province"], drop_first=True)
X = sm.add_constant(X).astype(float)

mnlogit_model = sm.MNLogit(y, X)
mnlogit_res = mnlogit_model.fit(disp=False)
print(mnlogit_res.summary())
print("\nNote: base outcome = Unemployed (category 0). Positive coefficient on")
print("distance_cbd_km for 'Formal' equation implies distance REDUCES relative")
print("odds of formal employment vs unemployment -- consistent with spatial mismatch.")

# -----------------------------------------------------------------
# 4. Panel fixed-effects model: Pr(Formal) on transport cost & distance
# -----------------------------------------------------------------
print("\n" + "=" * 80)
print("4. PANEL FIXED EFFECTS — linear probability model of formal employment")
print("=" * 80)

panel_df = df.set_index(["individual_id", "wave"])

# distance is time-invariant so it is absorbed by entity FE; interact with
# wave-varying transport cost instead, and include distance separately in a
# between/pooled specification for comparison.
fe_formula = "formal ~ transport_cost_share_income + education_years + age + EntityEffects + TimeEffects"
mod_fe = PanelOLS.from_formula(fe_formula, data=panel_df, drop_absorbed=True)
res_fe = mod_fe.fit(cov_type="clustered", cluster_entity=True)
print(res_fe)

print("\n-- Pooled OLS (includes time-invariant distance_cbd_km) for comparison --")
pooled_formula = ("formal ~ 1 + transport_cost_share_income + distance_cbd_km + "
                   "education_years + age + C(race) + C(gender) + C(province) + C(wave)")
res_pooled = smf.ols(pooled_formula, data=df).fit(
    cov_type="cluster", cov_kwds={"groups": df["individual_id"]})
print(res_pooled.summary())

# -----------------------------------------------------------------
# 5. IV / 2SLS: distance instruments for transport cost
# -----------------------------------------------------------------
print("\n" + "=" * 80)
print("5. IV (2SLS) — distance_cbd_km instruments transport_cost_share_income")
print("   Outcome: formal employment probability (linear probability IV)")
print("=" * 80)

iv_df = df.copy()
iv_df = sm.add_constant(iv_df, has_constant="add")
iv_formula = ("formal ~ 1 + education_years + age + C(race) + C(gender) + C(province) "
              "+ [transport_cost_share_income ~ distance_cbd_km]")
iv_res = IV2SLS.from_formula(iv_formula, data=iv_df).fit(cov_type="clustered",
                                                           clusters=iv_df["individual_id"])
print(iv_res)
print("\nFirst-stage relevance and Kain-hypothesis interpretation: distance is")
print("plausibly exogenous to current labor outcomes (apartheid-era settlement,")
print("fixed pre-determined residence) but strongly predicts transport cost burden,")
print("satisfying the relevance condition. Exclusion restriction (distance affects")
print("formal employment only through transport cost / access) should be argued")
print("carefully in any real-data application -- e.g. by controlling flexibly for")
print("local labor-demand conditions and neighborhood effects.")

# -----------------------------------------------------------------
# 6. Transition-specific logits (discrete-time hazard framing)
# -----------------------------------------------------------------
print("\n" + "=" * 80)
print("6. TRANSITION-SPECIFIC LOGITS")
print("=" * 80)

trans_df = df.dropna(subset=["prev_employment_status"]).copy()

print("\n-- (a) Among Informal at t-1: Pr(move to Formal) --")
sub_a = trans_df[trans_df["prev_employment_status"] == "Informal"].copy()
sub_a["to_formal"] = (sub_a["employment_status"] == "Formal").astype(int)
logit_a = smf.logit(
    "to_formal ~ distance_cbd_km + transport_cost_share_income + education_years + age + C(race) + C(gender)",
    data=sub_a).fit(disp=False, cov_type="cluster", cov_kwds={"groups": sub_a["individual_id"]})
print(logit_a.summary())

print("\n-- (b) Among Unemployed at t-1: Pr(move to Informal, vs staying unemployed) --")
sub_b = trans_df[trans_df["prev_employment_status"] == "Unemployed"].copy()
sub_b = sub_b[sub_b["employment_status"] != "Formal"]  # binary comparison informal vs unemployed
sub_b["to_informal"] = (sub_b["employment_status"] == "Informal").astype(int)
logit_b = smf.logit(
    "to_informal ~ distance_cbd_km + transport_cost_share_income + education_years + age + C(race) + C(gender)",
    data=sub_b).fit(disp=False, cov_type="cluster", cov_kwds={"groups": sub_b["individual_id"]})
print(logit_b.summary())

print("\n-- (c) Among Formal at t-1: Pr(exit to Informal/Unemployed) --")
sub_c = trans_df[trans_df["prev_employment_status"] == "Formal"].copy()
sub_c["exit_formal"] = (sub_c["employment_status"] != "Formal").astype(int)
logit_c = smf.logit(
    "exit_formal ~ distance_cbd_km + transport_cost_share_income + education_years + age + C(race) + C(gender)",
    data=sub_c).fit(disp=False, cov_type="cluster", cov_kwds={"groups": sub_c["individual_id"]})
print(logit_c.summary())

print("\nDone.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 28.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.9/118.9 kB 9.4 MB/s eta 0:00:00
1. DESCRIPTIVE STATISTICS
                               count         mean          std     min         25%       50%        75%        max
distance_cbd_km              12500.0    19.684336     9.220975   0.500    13.42750    20.070    26.2500     49.490
transport_cost_month_rand    12500.0  1594.653807   847.588240  25.680  1020.37000  1546.165  2108.9975   5859.800
transport_cost_share_income  12500.0     0.381932     0.186874   0.007     0.25575     0.382     0.5060      1.208
education_years              12500.0     9.889200     2.759907   1.900     8.00000     9.700    11.6000     20.000
monthly_earnings_rand        12500.0  3632.178418  3854.504121   0.000     0.00000  2825.445  6382.8400  20135.920

Employment status shares by wave:
employment_status  Formal  Informal  Unemployed
wave                                        

/tmp/ipykernel_551/759904869.py:107: AbsorbingEffectWarning: 
Variables have been fully absorbed and have removed from the regression:

education_years, age

  res_fe = mod_fe.fit(cov_type="clustered", cluster_entity=True)


                          PanelOLS Estimation Summary                           
Dep. Variable:                 formal   R-squared:                        0.0005
Estimator:                   PanelOLS   R-squared (Between):             -0.2545
No. Observations:               12500   R-squared (Within):               0.0030
Date:                Sun, Aug 16 2026   R-squared (Overall):             -0.1623
Time:                        19:01:04   Log-likelihood                   -3443.6
Cov. Estimator:             Clustered                                           
                                        F-statistic:                      5.3876
Entities:                        2500   P-value                           0.0203
Avg Obs:                       5.0000   Distribution:                  F(1,9995)
Min Obs:                       5.0000                                           
Max Obs:                       5.0000   F-statistic (robust):             4.2133
                            